In [48]:
import os

import pandas as pd
import mlflow
import psycopg
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from dotenv import load_dotenv
from sklearn.preprocessing import (
    OneHotEncoder, 
    SplineTransformer, 
    QuantileTransformer, 
    RobustScaler,
    PolynomialFeatures,
    KBinsDiscretizer,
)

load_dotenv("/home/mle-user/mle_projects/mle-mlflow/.env")

TABLE_NAME = "users_churn"

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

EXPERIMENT_NAME = "users_churn_14"
RUN_NAME = "preprocessing" 
REGISTRY_MODEL_NAME = "churn_model"

In [49]:
connection = {"sslmode": "require", "target_session_attrs": "read-write"}
postgres_credentials = {
    "host": os.getenv("DB_DESTINATION_HOST"),
    "port": os.getenv("DB_DESTINATION_PORT"),
    "dbname": os.getenv("DB_DESTINATION_NAME"),
    "user": os.getenv("DB_DESTINATION_USER"),
    "password": os.getenv("DB_DESTINATION_PASSWORD"),
}

connection.update(postgres_credentials)

with psycopg.connect(**connection) as conn:

    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {TABLE_NAME}")
        data = cur.fetchall()
        columns = [col[0] for col in cur.description]

df = pd.DataFrame(data, columns=columns)

df.head(2)

,id,customer_id,begin_date,end_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,...,device_protection,tech_support,streaming_tv,streaming_movies,gender,senior_citizen,partner,dependents,multiple_lines,target
0,11,9763-GRSKD,2019-01-01,NaT,Month-to-month,Yes,Mailed check,49.95,587.45,DSL,...,No,No,No,No,Male,0,Yes,Yes,No,0
1,23,1066-JKSGK,2019-11-01,2019-12-01,Month-to-month,No,Mailed check,20.15,20.15,None,...,None,None,None,None,Male,0,No,No,No,1


In [50]:
obj_df = df.select_dtypes(include="object")
obj_df

,customer_id,type,paperless_billing,payment_method,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,gender,partner,dependents,multiple_lines
0,9763-GRSKD,Month-to-month,Yes,Mailed check,DSL,Yes,No,No,No,No,No,Male,Yes,Yes,No
1,1066-JKSGK,Month-to-month,No,Mailed check,None,None,None,None,None,None,None,Male,No,No,No
2,3638-WEABW,Two year,Yes,Credit card (automatic),DSL,No,Yes,No,Yes,No,No,Female,Yes,No,Yes
3,6322-HRPFA,Month-to-month,No,Credit card (automatic),DSL,Yes,Yes,No,Yes,No,No,Male,Yes,Yes,No
4,6865-JZNKO,Month-to-month,Yes,Bank transfer (automatic),DSL,Yes,Yes,No,No,No,No,Female,No,No,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0376-OIWME,Month-to-month,Yes,Electronic check,Fiber optic,Yes,No,No,No,Yes,Yes,Male,Yes,No,No
7039,3585-ISXZP,Month-to-month,No,Bank transfer (automatic),Fiber optic,No,No,No,No,Yes,Yes,Female,No,No,Yes
7040,0218-QNVAS,One year,No,Bank transfer (automatic),Fiber optic,No,Yes,No,No,Yes,Yes,Male,Yes,Yes,Yes
7041,6583-QGCSI,Month-to-month,Yes,Electronic check,Fiber optic,No,Yes,No,No,Yes,No,Female,Yes,No,Yes


In [51]:
from sklearn.preprocessing import OneHotEncoder

# определение категориальных колонок, которые будут преобразованы
cat_columns = ["type", "payment_method", "internet_service", "gender"]

encoder_oh = OneHotEncoder(
    categories='auto', 
    handle_unknown='ignore',
    max_categories=10,
    sparse_output=False,
    drop="first"
)

# применение OneHotEncoder к данным. Преобразование категориальных данных в массив
encoded_features = encoder_oh.fit_transform(df[cat_columns].to_numpy())

new_columns = encoder_oh.get_feature_names_out(cat_columns)

encoded_df = pd.DataFrame(encoded_features, columns=new_columns)

# конкатенация исходного DataFrame с новым DataFrame, содержащим закодированные категориальные признаки
# axis=1 означает конкатенацию по колонкам
obj_df = pd.concat([obj_df, encoded_df], axis=1)

obj_df.head(2)

,customer_id,type,paperless_billing,payment_method,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,...,dependents,multiple_lines,type_One year,type_Two year,payment_method_Credit card (automatic),payment_method_Electronic check,payment_method_Mailed check,internet_service_Fiber optic,internet_service_None,gender_Male
0,9763-GRSKD,Month-to-month,Yes,Mailed check,DSL,Yes,No,No,No,No,...,Yes,No,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1,1066-JKSGK,Month-to-month,No,Mailed check,None,None,None,None,None,None,...,No,No,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0


In [52]:
from sklearn.preprocessing import SplineTransformer, QuantileTransformer, RobustScaler, PolynomialFeatures, KBinsDiscretizer


num_columns = ["monthly_charges", "total_charges"]

df[num_columns] = df[num_columns].fillna(0)

num_df = pd.DataFrame(index=df.index)

n_knots = 3
degree_spline = 4
n_quantiles=100
degree = 3
n_bins = 5
encode = 'ordinal'
strategy = 'uniform'
subsample = None


# SplineTransformer
encoder_spl = SplineTransformer(n_knots=n_knots, degree=degree_spline)
encoded_features = encoder_spl.fit_transform(df[num_columns].to_numpy())

encoded_df = pd.DataFrame(
    encoded_features, 
    columns=encoder_spl.get_feature_names_out(num_columns)
)
num_df = pd.concat([num_df, encoded_df], axis=1)


# QuantileTransformer
encoder_q = QuantileTransformer(n_quantiles=n_quantiles)
encoded_features = encoder_q.fit_transform(df[num_columns].to_numpy())

encoded_df = pd.DataFrame(
    encoded_features, 
    columns=encoder_q.get_feature_names_out(num_columns)
)
encoded_df.columns = [col + f"_q_{n_quantiles}" for col in num_columns]
num_df = pd.concat([num_df, encoded_df], axis=1)


# RobustScaler
encoder_rb = RobustScaler()
encoded_features = encoder_rb.fit_transform(df[num_columns].to_numpy())

encoded_df = pd.DataFrame(
    encoded_features, 
    columns=encoder_rb.get_feature_names_out(num_columns)
)
encoded_df.columns = [col + f"_robust" for col in num_columns]
num_df = pd.concat([num_df, encoded_df], axis=1)


# PolynomialFeatures
encoder_pol = PolynomialFeatures(degree=degree)
encoded_features = encoder_pol.fit_transform(df[num_columns].to_numpy())

encoded_df = pd.DataFrame(
    encoded_features, 
    columns=encoder_pol.get_feature_names_out(num_columns)
)
# get all columns after the intercept and original features
encoded_df.columns = [col + f"_poly" for col in encoded_df.columns]
num_df = pd.concat([num_df, encoded_df], axis=1)


# KBinsDiscretizer
encoder_kbd = KBinsDiscretizer(n_bins=n_bins, encode=encode, strategy=strategy, subsample=subsample)
encoded_features = encoder_kbd.fit_transform(df[num_columns].to_numpy())

encoded_df = pd.DataFrame(
    encoded_features, 
    columns=encoder_kbd.get_feature_names_out(num_columns)
)
encoded_df.columns = [col + f"_bin" for col in num_columns]
num_df = pd.concat([num_df, encoded_df], axis=1)




num_df.head(2)


,monthly_charges_sp_0,monthly_charges_sp_1,monthly_charges_sp_2,monthly_charges_sp_3,monthly_charges_sp_4,monthly_charges_sp_5,total_charges_sp_0,total_charges_sp_1,total_charges_sp_2,total_charges_sp_3,...,total_charges_poly,monthly_charges^2_poly,monthly_charges total_charges_poly,total_charges^2_poly,monthly_charges^3_poly,monthly_charges^2 total_charges_poly,monthly_charges total_charges^2_poly,total_charges^3_poly,monthly_charges_bin,total_charges_bin
0,0.000774,0.142550,0.588331,0.261746,6.599052e-03,0.0,0.023296,0.387299,0.520245,0.069146,...,587.45,2495.0025,29343.1275,345097.5025,124625.374875,1.465689e+06,1.723762e+07,2.027275e+08,1.0,0.0
1,0.035713,0.439097,0.476855,0.048335,8.516456e-08,0.0,0.040899,0.456008,0.460648,0.042445,...,20.15,406.0225,406.0225,406.0225,8181.353375,8.181353e+03,8.181353e+03,8.181353e+03,0.0,0.0


In [54]:
numeric_transformer = ColumnTransformer(transformers=[('spl', encoder_spl, num_columns), ('q', encoder_q, num_columns), ('rb', encoder_rb, num_columns), ('pol', encoder_pol, num_columns), ('kbd', encoder_kbd, num_columns)])

categorical_transformer = Pipeline(steps=[('encoder', encoder_oh)])

preprocessor = ColumnTransformer(transformers=[
('num', numeric_transformer, num_columns), 
('cat', categorical_transformer, cat_columns)], n_jobs=-1)

encoded_features = preprocessor.fit_transform(df)

transformed_df = pd.DataFrame(encoded_features, columns=preprocessor.get_feature_names_out())

df = pd.concat([df, transformed_df], axis=1)

df.head(2)

,id,customer_id,begin_date,end_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,...,num__kbd__monthly_charges,num__kbd__total_charges,cat__type_One year,cat__type_Two year,cat__payment_method_Credit card (automatic),cat__payment_method_Electronic check,cat__payment_method_Mailed check,cat__internet_service_Fiber optic,cat__internet_service_None,cat__gender_Male
0,11,9763-GRSKD,2019-01-01,NaT,Month-to-month,Yes,Mailed check,49.95,587.45,DSL,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1,23,1066-JKSGK,2019-11-01,2019-12-01,Month-to-month,No,Mailed check,20.15,20.15,None,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0


In [57]:
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ['AWS_ACCESS_KEY_ID'] = os.getenv('S3_ACCESS_KEY')
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("S3_SECRET_KEY")

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is not None:
    experiment_id = experiment.experiment_id
else:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    mlflow.sklearn.log_model(preprocessor, "column_transformer")

2026/06/14 16:06:09 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


In [59]:
run_id

'0b9f5b55dd2b4288964e259eb9c1710e'

In [ ]:
import os
import pandas as pd
from catboost import CatBoostClassifier
import mlflow
from dotenv import load_dotenv
from sklearn.metrics import roc_auc_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import SplineTransformer, QuantileTransformer, RobustScaler, PolynomialFeatures, KBinsDiscretizer, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

cat_columns = ["type", "payment_method", "internet_service", "gender"]
num_columns = ["monthly_charges", "total_charges"]

load_dotenv("/home/mle-user/mle_projects/mle-mlflow/.env")

EXPERIMENT_NAME = "churn_prediction"
RUN_NAME = "new_fitch_V1"
REGISTRY_MODEL_NAME = "churn_model_georgioparin"

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://yandexcloud.net"
os.environ['AWS_ACCESS_KEY_ID'] = os.getenv('S3_ACCESS_KEY')
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("S3_SECRET_KEY")

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

df = pd.read_csv('~/mle_projects/mle-dvc/data/initial_data.csv')
df[num_columns] = df[num_columns].fillna(0)

encoder_spl = SplineTransformer(n_knots=3, degree=4)
encoder_q = QuantileTransformer(n_quantiles=100)
encoder_rb = RobustScaler()
encoder_pol = PolynomialFeatures(degree=3)
encoder_kbd = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='uniform')
encoder_oh = OneHotEncoder(categories='auto', handle_unknown='ignore', max_categories=10, sparse_output=False, drop="first")

numeric_transformer = ColumnTransformer(transformers=[
    ('spl', encoder_spl, num_columns), 
    ('q', encoder_q, num_columns), 
    ('rb', encoder_rb, num_columns), 
    ('pol', encoder_pol, num_columns), 
    ('kbd', encoder_kbd, num_columns)
])

categorical_transformer = Pipeline(steps=[('encoder', encoder_oh)])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_columns), 
    ('cat', categorical_transformer, cat_columns)
], n_jobs=-1)

encoded_features = preprocessor.fit_transform(df)
transformed_df = pd.DataFrame(encoded_features, columns=preprocessor.get_feature_names_out(), index=df.index)

df_final = pd.concat([df, transformed_df], axis=1)

y = df_final['target']
columns_to_drop = ['id', 'begin_date', 'end_date', 'target']
X = df_final.drop(columns=[col for col in columns_to_drop if col in df_final.columns])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cat_cols = list(X.select_dtypes(include=['object']).columns)

model = CatBoostClassifier(iterations=150, cat_features=cat_cols, verbose=0)
model.fit(X_train, y_train)

prediction = model.predict(X_test)
probas = model.predict_proba(X_test)[:, 1]

metrics = {
    "auc": roc_auc_score(y_test, probas),
    "precision": precision_score(y_test, prediction),
    "recall": recall_score(y_test, prediction)
}

pip_requirements = "/home/mle-user/mle_projects/mle-mlflow/requirements.txt"
signature = mlflow.models.infer_signature(X_test, prediction)
input_example = X_test[:10]
metadata = {'model_type': 'monthly'}

experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    mlflow.log_metrics(metrics)

    model_info = mlflow.catboost.log_model(
        cb_model=model,
        artifact_path="models",
        pip_requirements=pip_requirements,
        signature=signature,
        input_example=input_example,
        metadata=metadata,
        registered_model_name=REGISTRY_MODEL_NAME,
        await_registration_for=60
    )

/home/mle-user/.local/lib/python3.10/site-packages/sklearn/preprocessing/_discretization.py:248: FutureWarning: In version 1.5 onwards, subsample=200_000 will be used by default. Set subsample explicitly to silence this warning in the mean time. Set subsample=None to disable subsampling explicitly.
  warnings.warn(
/home/mle-user/.local/lib/python3.10/site-packages/mlflow/models/signature.py:212: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/mod

EndpointConnectionError: Could not connect to the endpoint URL: "https://yandexcloud.net/s3-student-mle-20260422-ac16c1f877-freetrack/5/73adf585171c4c97b17c243d8b58f492/artifacts/models/model.cb"

In [14]:
import os
from mlflow import MlflowClient

# Настройки подключения к вашему серверу
TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000
os.environ["MLFLOW_TRACKING_URI"] = f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}"

# Сброс прокси на всякий случай
os.environ.pop("HTTP_PROXY", None)
os.environ.pop("HTTPS_PROXY", None)

client = MlflowClient()

# Название вашей модели
model_registred_name = "churn_model_georgioparin"

# 1. Получаем список последних версий модели
latest_versions = client.get_latest_versions(name=model_registred_name)

# 2. Берем самую последнюю из них
last_version_metadata = latest_versions[-1]

# 3. Заполняем ваши переменные
model_version_id = last_version_metadata.version
run_id = last_version_metadata.run_id

# Проверяем результат
print(f"model_registred_name = '{model_registred_name}'")
print(f"model_version_id = {model_version_id}")
print(f"run_id = '{run_id}'")


model_registred_name = 'churn_model_georgioparin'
model_version_id = 1
run_id = '815b2af78f9042438dcfe36624069397'
